# 01 — ENSO and IOD classification from OISST

- Niño3.4 and DMI (west − east box) monthly anomalies relative to `cfg.BASELINE`; optionally linearly
  detrended (`cfg.INDEX_DETREND`).
- 3-month running mean (centred). An El Niño (La Niña) **event** is ≥ `ENSO_MIN_MONTHS` consecutive
  months with |N3.4| ≥ `ENSO_THRESH`.
- **ENSO year** Y = Jun(Y)–May(Y+1). Phase is set by whether the DJF value (3-month mean centred on
  January Y+1) lies inside an event. **Strong** El Niño: event peak ≥ `STRONG_THRESH`.
- IOD flag from the SON(Y) mean DMI vs `IOD_THRESH`.

Output: `enso_years.csv`, `indices_monthly.nc`.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import ausenso as cfg
warnings.filterwarnings("ignore", category=RuntimeWarning)
cfg.ensure_dirs()
print(dict(baseline=cfg.BASELINE, detrend=cfg.INDEX_DETREND, thresh=cfg.ENSO_THRESH,
           min_months=cfg.ENSO_MIN_MONTHS, strong=cfg.STRONG_THRESH, iod=cfg.IOD_THRESH))

In [ ]:
daily = xr.open_dataset(cfg.F_INDICES_DAILY)
mon = daily.resample(time="MS").mean()
ndays = daily.nino34.resample(time="MS").count()
mon = mon.where(ndays == ndays.time.dt.days_in_month, drop=True)   # complete months only
print(mon.time.values[[0, -1]])

In [ ]:
base = mon.sel(time=slice(str(cfg.BASELINE[0]), str(cfg.BASELINE[1])))
anom = mon.groupby("time.month") - base.groupby("time.month").mean()
anom = anom.drop_vars("month")
if cfg.INDEX_DETREND:
    fit = anom.polyfit("time", deg=1)
    for v in anom.data_vars:
        trend = xr.polyval(anom.time, fit[f"{v}_polyfit_coefficients"])
        print(f"{v} trend: {float(trend[-1] - trend[0]):.2f} °C over record")
        anom[v] = anom[v] - (trend - trend.mean())
n34 = anom.nino34.rolling(time=3, center=True).mean()
dmi = anom.dmi_w - anom.dmi_e

In [ ]:
def runs(mask):
    """(start, end) index pairs of True runs."""
    m = np.r_[False, np.asarray(mask), False].astype(int)
    d = np.diff(m)
    return list(zip(np.where(d == 1)[0], np.where(d == -1)[0] - 1))

v = n34.values
event_id = np.full(v.size, -1)
events = []
for sign, ph in [(1, "EN"), (-1, "LN")]:
    for s, e in runs(np.nan_to_num(sign * v) >= cfg.ENSO_THRESH):
        if e - s + 1 >= cfg.ENSO_MIN_MONTHS:
            seg = v[s:e + 1]
            k = len(events)
            event_id[s:e + 1] = k
            events.append(dict(phase=ph, start=n34.time.values[s], end=n34.time.values[e],
                               peak=float(seg[np.argmax(sign * seg)]),
                               peak_time=n34.time.values[s + np.argmax(sign * seg)]))
events = pd.DataFrame(events).sort_values("start").reset_index(drop=True)
events

In [ ]:
times = pd.DatetimeIndex(n34.time.values)
rows = []
for y in range(cfg.ENSO_YEARS[0], cfg.ENSO_YEARS[1] + 1):
    i = times.get_loc(pd.Timestamp(f"{y + 1}-01-01"))
    eid = event_id[i]
    son = dmi.sel(time=slice(f"{y}-09-01", f"{y}-11-30")).mean().item()
    if eid >= 0:
        t0 = n34.time.values[i]
        ev = events[(events.start <= t0) & (events.end >= t0)].iloc[0]
        phase, peak = ev.phase, ev.peak
    else:
        phase, peak = "N", np.nan
    rows.append(dict(enso_year=y, label=f"{y}/{str(y + 1)[-2:]}", n34_DJF=round(float(v[i]), 2),
                     phase=phase, peak=round(peak, 2) if np.isfinite(peak) else np.nan,
                     strong_EN=bool(phase == "EN" and peak >= cfg.STRONG_THRESH),
                     dmi_SON=round(son, 2),
                     iod="+" if son >= cfg.IOD_THRESH else ("-" if son <= -cfg.IOD_THRESH else "0")))
table = pd.DataFrame(rows).set_index("enso_year")
table.to_csv(cfg.F_ENSO_TABLE)
pd.set_option("display.max_rows", 60)
table

In [ ]:
for nm, yrs in cfg.phase_years(table).items():
    print(f"{nm:17s} n={len(yrs):2d}: {[table.label[y] for y in yrs]}")
print()
print("El Niño years with +IOD:", [table.label[y] for y in table.index[(table.phase == 'EN') & (table.iod == '+')]])
print("La Niña years with -IOD:", [table.label[y] for y in table.index[(table.phase == 'LN') & (table.iod == '-')]])

Check the strong-El Niño list against known events (1982/83, 1997/98, 2015/16, 2023/24 are the usual
candidates). Membership is sensitive to `STRONG_THRESH`, detrending and the OISST-based index, so
inspect the `peak` column before fixing the definition.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
ax[0].fill_between(n34.time, 0, n34, where=n34 > 0, color="C3", lw=0)
ax[0].fill_between(n34.time, 0, n34, where=n34 < 0, color="C0", lw=0)
for th in (cfg.ENSO_THRESH, -cfg.ENSO_THRESH, cfg.STRONG_THRESH):
    ax[0].axhline(th, color="k", lw=0.5, ls=":")
for y in table.index[table.strong_EN]:
    ax[0].axvspan(pd.Timestamp(f"{y}-06-01"), pd.Timestamp(f"{y + 1}-05-31"), color="0.85", zorder=0)
ax[0].set_ylabel("Niño3.4 (°C, 3-mo)")
ax[1].plot(dmi.time, dmi.rolling(time=3, center=True).mean(), "k", lw=0.8)
ax[1].axhline(cfg.IOD_THRESH, color="k", lw=0.5, ls=":"); ax[1].axhline(-cfg.IOD_THRESH, color="k", lw=0.5, ls=":")
ax[1].set_ylabel("DMI (°C, 3-mo)")
fig.savefig(cfg.FIG / "01_indices.png", dpi=200, bbox_inches="tight")

In [ ]:
xr.Dataset({"nino34": anom.nino34, "nino34_3mo": n34, "dmi": dmi}).to_netcdf(cfg.F_INDICES_MONTHLY)